---

## Copernicus Extension (`.cope`)

The `.cope` xarray accessor converts downloaded ERA5-Land data into pandas DataFrames with aggregated weather variables per ADM2.

In [ ]:
import dotenv
dotenv.load_dotenv()

In [ ]:
from satellite.request import ReanalysisERA5Land
from satellite import ADM2

### Download and convert to DataFrame

In [ ]:
req = ReanalysisERA5Land(locale='BRA', date='2024-10-22')
ds = req.run('cope_single')

adm = ADM2.get(code='3304557', adm0='BRA')
df = ds.cope.to_dataframe(adm)
df

### Multiple ADM2s

In [ ]:
from satellite.geo import functional

with functional.session() as session:
    codes = [r[0] for r in session.execute(
        "SELECT code FROM adm2 WHERE adm0 = 'BRA' LIMIT 5"
    ).fetchall()]

adms = [ADM2.get(code=c, adm0='BRA') for c in codes]
df = ds.cope.to_dataframe(adms)
print(f'Shape: {df.shape}, unique geocodes: {df["geocode"].nunique()}')
df.head(10)

### `adm_ds` — aggregated xarray Dataset per ADM2

In [ ]:
adm_ds = ds.cope.adm_ds(ADM2.get(code="3304557", adm0="BRA"))
adm_ds

In [ ]:
import matplotlib.pyplot as plt

plt.bar([str(adm_ds.time.values[0])[:10]], adm_ds.temp_med.values)
plt.title("Temperature (Rio de Janeiro)")
plt.ylabel("Temperature (°C)")
plt.show()

### Multiple days with precipitation correction

In [ ]:
req = ReanalysisERA5Land(
    locale='BRA',
    date='2024-10-18/2024-10-22',
    variable=['2m_temperature', 'total_precipitation'],
    time=['00:00', '06:00', '12:00', '18:00'],
)
ds = req.run('cope_range')

adm = ADM2.get(code='3304557', adm0='BRA')
df = ds.cope.to_dataframe(adm)
df